# 🚦 ITS Phase 1 Demo: Traffic Generation

This notebook demonstrates the traffic generation system for the Intelligent Transportation System (ITS) project.

## What This Demo Covers

1. **Parse SUMO Network** - Load and analyze the road network
2. **Generate Traffic Scenarios** - Create synthetic traffic with different congestion levels
3. **Visualize Data** - Plot speed distributions and network statistics
4. **Prepare for GNN** - Format data for prediction model

---

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Import project modules
from src.sumo_integration.sumo_parser import SUMONetworkParser
from src.data_generation.traffic_generator import TrafficGenerator
from src.data_generation.speed_history_generator import SpeedHistoryGenerator

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Imports successful!")

## Step 1: Parse SUMO Network

First, we load and analyze the SUMO network file.

In [ ]:
# Parse network
net_file = "../data/sumo/map.net.xml"
parser = SUMONetworkParser(net_file)
parser.print_summary()

# Get statistics
stats = parser.get_network_stats()
print(f"\nNetwork Statistics:")
for key, value in stats.items():
    if key != 'network_bounds':
        print(f"  {key}: {value}")

In [ ]:
# Visualize network topology
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Node locations
node_x = [node.x for node in parser.nodes.values()]
node_y = [node.y for node in parser.nodes.values()]
ax1.scatter(node_x, node_y, c='red', s=100, alpha=0.6, edgecolors='black')
ax1.set_title('Network Junctions', fontsize=14, fontweight='bold')
ax1.set_xlabel('X Coordinate')
ax1.set_ylabel('Y Coordinate')
ax1.grid(True, alpha=0.3)

# Plot 2: Edge length distribution
edge_lengths = [edge.length for edge in parser.edges.values()]
ax2.hist(edge_lengths, bins=20, edgecolor='black', alpha=0.7)
ax2.set_title('Road Segment Length Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Length (meters)')
ax2.set_ylabel('Frequency')
ax2.axvline(np.mean(edge_lengths), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(edge_lengths):.1f}m')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2: Generate Traffic Scenarios

Now we'll generate traffic scenarios with different congestion levels.

In [ ]:
# Configuration for traffic generator
config = {
    'emergency_vehicle_ratio': 0.05,
    'speed_generation': {
        'timestep_minutes': 5,
        'noise_std': 3.0,
        'min_speed_factor': 0.1,
        'temporal_smoothing': 0.8
    }
}

# Create generator
generator = TrafficGenerator(parser, config)
print("✓ Traffic generator initialized")

In [ ]:
# Generate a normal scenario
scenario = generator.generate_traffic_scenario(
    num_vehicles=500,
    congestion_level=0.4,
    scenario_type="normal",
    output_dir="../data/generated"
)

## Step 3: Analyze Generated Data

Let's examine the generated traffic data.

In [ ]:
# Load generated data
with open('../data/generated/vehicles.json', 'r') as f:
    vehicles = json.load(f)

with open('../data/generated/edge_states.json', 'r') as f:
    edge_states = json.load(f)

print(f"Total vehicles: {len(vehicles)}")
print(f"Emergency vehicles: {sum(1 for v in vehicles if v['vehicle_type'] == 'emergency')}")
print(f"Normal vehicles: {sum(1 for v in vehicles if v['vehicle_type'] == 'normal')}")
print(f"\nEdges with traffic: {len(edge_states)}")

In [ ]:
# Visualize traffic distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Vehicle departure times
depart_times = [v['depart_time'] for v in vehicles]
axes[0, 0].hist(depart_times, bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Vehicle Departure Times', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Time (seconds)')
axes[0, 0].set_ylabel('Number of Vehicles')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Speed distribution across edges
speeds = [es['current_speed'] for es in edge_states.values()]
axes[0, 1].hist(speeds, bins=20, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Current Speed Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Speed (km/h)')
axes[0, 1].set_ylabel('Number of Edges')
axes[0, 1].axvline(np.mean(speeds), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(speeds):.1f} km/h')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Congestion levels
congestion = [es['congestion_factor'] for es in edge_states.values()]
axes[1, 0].hist(congestion, bins=20, edgecolor='black', alpha=0.7, color='red')
axes[1, 0].set_title('Congestion Factor Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Congestion Factor (0-1)')
axes[1, 0].set_ylabel('Number of Edges')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Vehicle count per edge
vehicle_counts = [es['vehicle_count'] for es in edge_states.values()]
axes[1, 1].hist(vehicle_counts, bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].set_title('Vehicles per Edge Distribution', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Number of Vehicles')
axes[1, 1].set_ylabel('Number of Edges')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: Examine Speed History

Visualize the generated speed history (for GNN input).

In [ ]:
# Load speed history
with open('../data/generated/speed_history.json', 'r') as f:
    speed_history = json.load(f)

# Load speed matrix
speed_matrix = np.load('../data/generated/speed_matrix.npy')
print(f"Speed matrix shape: {speed_matrix.shape}")
print(f"  - Timesteps: {speed_matrix.shape[0]} (each = 5 minutes)")
print(f"  - Edges/Nodes: {speed_matrix.shape[1]}")

In [ ]:
# Visualize speed time series for sample edges
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

edge_ids = list(speed_history.keys())[:3]  # First 3 edges

for i, edge_id in enumerate(edge_ids):
    data = speed_history[edge_id]
    times = data['timestamps']
    speeds = data['speeds']
    
    axes[i].plot(times, speeds, marker='o', linewidth=2, markersize=6)
    axes[i].set_title(f'Speed History: {edge_id}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Time (minutes)')
    axes[i].set_ylabel('Speed (km/h)')
    axes[i].grid(True, alpha=0.3)
    axes[i].axhline(np.mean(speeds), color='red', linestyle='--', alpha=0.5, label=f'Mean: {np.mean(speeds):.1f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of speed matrix
plt.figure(figsize=(14, 8))
sns.heatmap(speed_matrix.T[:20, :], cmap='RdYlGn', cbar_kws={'label': 'Speed (km/h)'}, vmin=0, vmax=60)
plt.title('Speed Heatmap (First 20 Edges)', fontsize=14, fontweight='bold')
plt.xlabel('Timestep (5-minute intervals)')
plt.ylabel('Edge/Node ID')
plt.tight_layout()
plt.show()

## Step 5: Compare Different Congestion Levels

Generate and compare scenarios with varying congestion.

In [ ]:
# Generate multiple scenarios
scenarios = {}
congestion_levels = [0.1, 0.3, 0.5, 0.7, 0.9]

for cong in congestion_levels:
    print(f"\n Generating scenario with congestion = {cong}...")
    scenario = generator.generate_traffic_scenario(
        num_vehicles=500,
        congestion_level=cong,
        scenario_type="normal",
        output_dir=f"../data/generated/scenario_{int(cong*100)}"
    )
    scenarios[cong] = scenario

In [ ]:
# Compare average speeds across scenarios
avg_speeds = {}
for cong, scenario in scenarios.items():
    speeds = [es['current_speed'] for es in scenario['edge_states'].values()]
    avg_speeds[cong] = np.mean(speeds)

plt.figure(figsize=(10, 6))
plt.plot(list(avg_speeds.keys()), list(avg_speeds.values()), marker='o', linewidth=3, markersize=10)
plt.title('Average Network Speed vs Congestion Level', fontsize=14, fontweight='bold')
plt.xlabel('Congestion Level', fontsize=12)
plt.ylabel('Average Speed (km/h)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nAverage speeds:")
for cong, speed in avg_speeds.items():
    print(f"  Congestion {cong:.1f}: {speed:.1f} km/h")

## Summary

✅ **Phase 1 Complete!**

We have successfully:
1. Parsed SUMO network topology
2. Generated synthetic traffic scenarios
3. Created speed history data for GNN
4. Visualized traffic patterns
5. Prepared data for the next phases

### Next Steps:

**Phase 2: GNN Prediction**
- Load trained GNN model
- Predict future speeds
- Map predictions to SUMO edges

**Phase 3: Routing Engine**
- Implement A* and Dijkstra
- Generate optimal routes
- Apply traffic constraints

**Phase 4: SUMO Simulation**
- Create route files
- Launch SUMO-GUI
- Visualize traffic flow

---

**Generated Files Location:**  
`../data/generated/`

Use these files as input for subsequent phases!